# Clase 08: Tipos y fuentes de datos

**Objetivos de aprendizaje:**
- Clasificar un dato como estructurado, semiestructurado o no estructurado.
- Identificar fuentes de datos comunes (bases de datos, APIs, web scraping, streams).
- Leer CSV/JSON con pandas.

**Duración estimada:** 110 min

**Capítulos del manual:**
- [09 — Tipos de datos](../manual/09_tipos_de_datos.md)
- [10 — De dónde vienen los datos](../manual/10_fuentes_de_datos_apis.md)


## Parte 1: Tipos de datos

Antes de analizar cualquier dato, necesitas saber con qué tipo de dato estás tratando. Hay tres categorías:

- **Estructurados**: esquema fijo, formato tabular (SQL, hojas de cálculo, sensores GPS).
- **Semiestructurados**: organización parcial vía etiquetas/metadatos, sin esquema fijo (XML, JSON, emails).
- **No estructurados**: sin esquema identificable (texto libre, imágenes, audio, vídeo, PDFs).

Vamos a construir un ejemplo de cada uno y ver cómo cambia la forma de trabajar con cada tipo.

In [ ]:
import pandas as pd
import numpy as np
import json
import io
import sqlite3
from collections import Counter

# --- 1. Datos ESTRUCTURADOS ---
ventas_estructuradas = pd.DataFrame({
    "id_pedido": [1, 2, 3, 4],
    "cliente": ["Ana", "Luis", "Marta", "Ana"],
    "importe": [59.90, 120.00, 15.50, 42.30],
})
print("--- Estructurados ---")
print(ventas_estructuradas)
print(ventas_estructuradas.dtypes)


In [ ]:
# --- 2. Datos SEMIESTRUCTURADOS ---
# Simulamos la respuesta típica de una API: JSON con estructura anidada
pedidos_json = '''
[
  {"id_pedido": 1, "cliente": "Ana", "productos": ["libro", "taza"]},
  {"id_pedido": 2, "cliente": "Luis", "productos": ["portatil"]},
  {"id_pedido": 3, "cliente": "Marta", "productos": ["libro", "cuaderno", "boli"]}
]
'''

pedidos_dict = json.loads(pedidos_json)
print("--- Semiestructurados (JSON crudo) ---")
print(pedidos_dict)

# json_normalize aplana la estructura a un DataFrame
pedidos_df = pd.json_normalize(pedidos_dict)
pedidos_df["n_productos"] = pedidos_df["productos"].apply(len)
print("\n--- Semiestructurados (como DataFrame) ---")
print(pedidos_df)


In [ ]:
# --- 3. Datos NO ESTRUCTURADOS ---
comentarios = [
    "Muy buen servicio, llego rapido",
    "El producto llego danado, pesima experiencia",
    "Precio correcto, volveria a comprar",
    "Rapido y muy buen precio, encantada",
]

print("--- No estructurados (texto libre) ---")
for c in comentarios:
    print("-", c)

# Antes de analizarlo con pandas hay que decidir QUÉ vamos a medir:
# aquí, contamos palabras por comentario y la frecuencia global
longitudes = [len(c.split()) for c in comentarios]
print("\nPalabras por comentario:", longitudes)

todas_las_palabras = " ".join(comentarios).lower().split()
frecuencia = Counter(todas_las_palabras)
print("Palabras más comunes:", frecuencia.most_common(5))


Fíjate en el esfuerzo creciente: los datos estructurados ya estaban listos; los semiestructurados necesitaron un paso de conversión (`json.loads` + `json_normalize`); los no estructurados necesitaron que decidiéramos nosotros mismos qué medir (aquí, contar palabras) antes de que se parecieran en algo a una tabla.

## Parte 2: ¿De dónde vienen los datos?

Ahora la otra mitad de la pregunta: ¿de dónde salen esos datos? Fuentes típicas:

- **Bases de datos relacionales internas**: SQL Server, Oracle, MySQL, IBM DB2.
- **Archivos planos y XML**: CSV (el más común), hojas de cálculo, XML (jerárquico).
- **APIs y servicios web**: redes sociales, mercados bursátiles, validación de datos — normalmente devuelven JSON.
- **Web scraping**: BeautifulSoup, Scrapy, Pandas, Selenium — cuando no hay una API disponible.
- **Streams de datos**: Apache Kafka, Spark Streaming, Apache Storm — datos que llegan de forma continua (sensores IoT, transacciones, tickers bursátiles).

Vamos a simular tres fuentes distintas —CSV, JSON de una API, y una base de datos relacional— y a combinarlas, que es lo que hace un data scientist en la práctica casi siempre.

In [ ]:
# --- Fuente 1: archivo plano (CSV) ---
csv_texto = '''id_cliente,nombre,ciudad,gasto_total
1,Ana,Madrid,340.50
2,Luis,Sevilla,120.00
3,Marta,Bilbao,89.90
'''
clientes_csv = pd.read_csv(io.StringIO(csv_texto))
print("--- Fuente: CSV ---")
print(clientes_csv)


In [ ]:
# --- Fuente 2: API (JSON) ---
json_texto = '''
[
  {"cp": "28001", "ciudad": "Madrid", "provincia": "Madrid"},
  {"cp": "41001", "ciudad": "Sevilla", "provincia": "Sevilla"},
  {"cp": "48001", "ciudad": "Bilbao", "provincia": "Bizkaia"}
]
'''
codigos_postales = pd.read_json(io.StringIO(json_texto))
print("--- Fuente: API (JSON) ---")
print(codigos_postales)


In [ ]:
# --- Fuente 3: base de datos relacional (sqlite3 en memoria) ---
conexion = sqlite3.connect(":memory:")
cursor = conexion.cursor()

cursor.execute('''
    CREATE TABLE pedidos (
        id_pedido INTEGER,
        id_cliente INTEGER,
        importe REAL
    )
''')
cursor.executemany(
    "INSERT INTO pedidos VALUES (?, ?, ?)",
    [(1, 1, 59.90), (2, 2, 120.00), (3, 1, 15.50), (4, 3, 42.30)],
)
conexion.commit()

pedidos_db = pd.read_sql_query("SELECT * FROM pedidos", conexion)
print("--- Fuente: base de datos relacional ---")
print(pedidos_db)

conexion.close()


In [ ]:
# --- Combinando las tres fuentes ---
combinado = pedidos_db.merge(clientes_csv, on="id_cliente")
resumen = combinado.groupby("ciudad")["importe"].sum()
print("--- Gasto total por ciudad, combinando BD + CSV ---")
print(resumen)


## Resumen de la clase

- **Estructurados** (esquema fijo, tablas), **semiestructurados** (JSON/XML, jerárquicos), **no estructurados** (texto libre, imágenes): cada uno exige una herramienta distinta.
- Los datos llegan de bases de datos relacionales, archivos planos, APIs, web scraping o streams en tiempo real.
- Da igual el origen: el trabajo del data scientist es siempre convertir los datos a un formato común (normalmente tabular) para poder cruzarlos y analizarlos, como hicimos combinando la base de datos con el CSV.
- `pandas.read_csv`, `pandas.read_json` y `pandas.read_sql_query` son las tres puertas de entrada más habituales a un DataFrame.

**Material relacionado:**
- Capítulos del manual: [09 — Tipos de datos](../manual/09_tipos_de_datos.md) y [10 — De dónde vienen los datos](../manual/10_fuentes_de_datos_apis.md)
- Notebook de práctica: [`clase08_practica_tipos_y_fuentes_de_datos.ipynb`](../notebooks_practica/clase08_practica_tipos_y_fuentes_de_datos.ipynb)
